# A/B 测试数据清洗

本笔记本对原始 A/B 测试数据进行清洗，为后续统计分析做准备。

处理步骤：
1. 加载并浏览原始数据
2. 检查缺失值与数据质量
3. 移除重复用户记录
4. 移除分组与页面不匹配的样本
5. 保存清洗后的数据集

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/ab_data.csv')
df.head()

In [ ]:
print('数据集形状:', df.shape)
print('\n缺失值:')
print(df.isnull().sum())
print('\n分组分布:')
print(df['group'].value_counts())
print('\n页面分布:')
print(df['landing_page'].value_counts())

## 重复用户移除

每个用户在实验中应只出现一次，重复记录会导致结果偏差。

In [ ]:
print('重复用户数:', df['user_id'].duplicated().sum())

df_clean = df.drop_duplicates(subset=['user_id'], keep='first')
print('去重后记录数:', df_clean.shape[0])

## 分组与页面对齐检查

在规范的 A/B 测试中：
- 对照组 (control) 应看到旧版页面 (old_page)
- 实验组 (treatment) 应看到新版页面 (new_page)

不匹配的样本说明数据存在质量问题，需要剔除。

In [ ]:
mismatch = df_clean[
    ((df_clean['group'] == 'control') & (df_clean['landing_page'] != 'old_page')) |
    ((df_clean['group'] == 'treatment') & (df_clean['landing_page'] != 'new_page'))
]
print('不匹配样本数:', len(mismatch))

df_final = df_clean[
    ((df_clean['group'] == 'control') & (df_clean['landing_page'] == 'old_page')) |
    ((df_clean['group'] == 'treatment') & (df_clean['landing_page'] == 'new_page'))
]
print('最终清洗后记录数:', df_final.shape[0])

In [ ]:
print('最终分组分布:')
print(df_final['group'].value_counts())
print('\n最终页面分布:')
print(df_final['landing_page'].value_counts())

In [ ]:
df_final.to_csv('../data/ab_data_cleaned.csv', index=False)
print('清洗后数据已保存至 ../data/ab_data_cleaned.csv')